# Long-Term Memory: Episodic, Semantic, Procedural

Three durable memory subtypes for an agent, implemented side-by-side on LangChain vector stores.

The three memory classes share one design: a small typed schema, an `add` / `record` / `register` method, and a `recall` / `query` / `lookup` method. The point of the notebook is to show that *what* you store and *how* you query is what distinguishes the three subtypes - the underlying primitive (a vector store or a dict) is mundane.

## Setup

Let's wire the model-agnostic stack and a single shared embedder.

In [1]:
# !pip install -q langchain langchain-google-genai langchain-openai langchain-anthropic langchain-community

from langchain.chat_models import init_chat_model
from langchain.embeddings import init_embeddings
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.documents import Document
from typing import List, Dict, Optional
from dataclasses import dataclass, field, asdict
from dotenv import load_dotenv
import os, json, uuid, datetime

load_dotenv()

True

In [2]:
# os.environ['GEMINI_API_KEY']  # the variable for API key

llm   = init_chat_model('gpt-4o-mini', model_provider='openai', temperature=0)
embed = init_embeddings('sentence-transformers/all-MiniLM-L6-v2', provider='huggingface')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


*Tip: every memory class below uses the same `embed` instance. In production you would namespace each store by tenant or user, but the embedder usually stays shared so retrieval scores are comparable across subtypes.*

## Episodic memory: agent history as an append-only log

Let's start with episodic memory - the simplest subtype to motivate. Episodic memory stores **specific past events**: what the agent did, when, and what happened. It is an append-only log keyed by run-id and timestamp.

Now, we will
- define an `Episode` record carrying `run_id`, `timestamp`, `query`, `action`, `outcome`,
- build an `EpisodicMemory` class wrapping an append-only list with `record(...)` and `recall(query, k)` methods,
- index every episode in an `InMemoryVectorStore` so similarity search complements recency-based recall,
- seed a few sample episodes from a fictitious support agent's history,
- recall episodes both by recency (the last 3) and by similarity ("what happened in run #42 about CSV exports?").

In [3]:
@dataclass
class Episode:
    """One past event in the agent's history.

    Episodes are IMMUTABLE - once written they are never edited. WHY: the value of an
    audit log is that it never lies; corrections live in NEW episodes that supersede."""
    run_id:    str
    timestamp: str
    query:     str
    action:    str
    outcome:   str

The `Episode` schema is deliberately small. The agent's `query` and `outcome` are the fields the embedder will index on - they are the most semantically discriminative parts of an event. `run_id` and `timestamp` stay structured so we can also recall by recency without going through the embedder.

In [4]:
class EpisodicMemory:
    """Append-only event log keyed by run-id / timestamp.

    Two recall modes are supported by design: RECENCY (the last k episodes) and SIMILARITY
    (the k most similar past episodes to a query). Production systems usually fall back from
    similarity to recency when the vector store is cold or empty."""

    def __init__(self, embedder):
        self._log: List[Episode] = []
        self._store = InMemoryVectorStore(embedding=embedder)

    def record(self, query: str, action: str, outcome: str, run_id: Optional[str] = None) -> Episode:
        """Append one episode and index it for similarity search."""
        ep = Episode(
            run_id=run_id or f'run-{len(self._log) + 1}',
            timestamp=datetime.datetime.utcnow().isoformat(),
            query=query,
            action=action,
            outcome=outcome,
        )
        self._log.append(ep)
        # WHY index query+outcome: they carry the semantically distinctive content.
        self._store.add_documents([Document(
            page_content=f'Q: {query}\nA: {action}\nOutcome: {outcome}',
            metadata={'run_id': ep.run_id, 'timestamp': ep.timestamp},
        )])
        return ep

    def recall(self, query: str, k: int = 3) -> List[Episode]:
        """Similarity recall over indexed episodes; falls back to recency if the log is empty."""
        if not self._log:
            return []
        hits = self._store.similarity_search(query, k=k)
        ids = [h.metadata['run_id'] for h in hits]
        by_id = {ep.run_id: ep for ep in self._log}
        return [by_id[i] for i in ids if i in by_id]

    def recent(self, k: int = 3) -> List[Episode]:
        """Recency-based recall - the last k episodes regardless of similarity."""
        return self._log[-k:]

Two methods, two question shapes. `recall(query, k)` answers *"have I seen something like this before?"*; `recent(k)` answers *"what was I just doing?"*. Most agents need both - similarity for grounding, recency for continuity.

In [5]:
episodic = EpisodicMemory(embed)

# Seed a small history for a support-agent style demo.
episodic.record(
    query='User cannot reset password on Safari',
    action='Sent magic-link email',
    outcome='User confirmed login after 2 minutes',
)
episodic.record(
    query='User reports CSV export produces empty file',
    action='Filed engineering ticket ENG-812',
    outcome='Hotfix shipped; user verified working',
)
episodic.record(
    query='User asked why invoice doubled',
    action='Explained proration on mid-cycle upgrade',
    outcome='User accepted explanation',
)
episodic.record(
    query='User requested SSO via Okta',
    action='Logged feature request to product backlog',
    outcome='Queued for next quarter',
)

print('=== RECENT EPISODES (last 2) ===')
for ep in episodic.recent(k=2):
    print(f'  [{ep.run_id}] {ep.query[:60]}... -> {ep.outcome[:50]}')

print('\n=== SIMILARITY RECALL: "CSV export issue" ===')
for ep in episodic.recall('CSV export issue', k=2):
    print(f'  [{ep.run_id}] {ep.query[:60]}... -> {ep.outcome[:50]}')

=== RECENT EPISODES (last 2) ===
  [run-3] User asked why invoice doubled... -> User accepted explanation
  [run-4] User requested SSO via Okta... -> Queued for next quarter

=== SIMILARITY RECALL: "CSV export issue" ===
  [run-2] User reports CSV export produces empty file... -> Hotfix shipped; user verified working
  [run-1] User cannot reset password on Safari... -> User confirmed login after 2 minutes


C:\Users\TusharSharma\AppData\Local\Temp\ipykernel_132\3997921477.py:16: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp=datetime.datetime.utcnow().isoformat(),


## Semantic memory: durable domain facts

Now let's build semantic memory. Semantic = facts about the world, NOT tied to any specific event. *"Our refund window is 30 days"* is semantic; *"u-42 asked about refunds on March 3"* is episodic. The access pattern is different too - semantic memory is **written rarely, queried often**.

Now, we will
- define a `SemanticMemory` class wrapping a vector store of curated domain facts,
- expose `add_fact(topic, fact, source)` for write and `query(question, k)` for read,
- seed a small Acme SaaS knowledge base (pricing, refund policy, integrations, support hours),
- query by intent ("how long do I have to return something?") and confirm semantic search outranks keyword overlap.

In [6]:
class SemanticMemory:
    """Curated knowledge base of durable, policy-like facts.

    Semantic memory is the agent's encyclopedia. Each fact is a self-contained sentence
    that should plausibly still be true tomorrow. WHY a vector store: questions rarely
    repeat the exact wording of stored facts, so similarity search beats keyword lookup."""

    def __init__(self, embedder):
        self._store = InMemoryVectorStore(embedding=embedder)
        self._facts: Dict[str, dict] = {}

    def add_fact(self, topic: str, fact: str, source: str = 'curated') -> None:
        """Upsert one fact under a stable topic key (so price changes are a one-line edit)."""
        self._facts[topic] = {'topic': topic, 'fact': fact, 'source': source}
        self._store.add_documents([Document(
            page_content=fact,
            metadata={'topic': topic, 'source': source},
        )])

    def query(self, question: str, k: int = 2) -> List[dict]:
        """Semantic search by question intent; returns the top-k matching facts."""
        hits = self._store.similarity_search(question, k=k)
        return [{'topic': h.metadata['topic'], 'fact': h.page_content,
                 'source': h.metadata.get('source', '')} for h in hits]

The `topic` slug is what makes upserts trivial. When pricing changes, you `add_fact('pricing:pro', ...)` again with the same key and the in-memory dict overwrites the prior entry. The vector store accumulates a new copy - in production you would replace by metadata filter, but for a teaching demo this stays out of the way.

In [7]:
semantic = SemanticMemory(embed)

ACME_FACTS = [
    ('pricing:starter',       'The Starter plan costs $9 per month and includes 3 seats.', 'pricing_page_v3'),
    ('pricing:pro',           'The Pro plan costs $49 per seat per month and includes SSO and priority email support.', 'pricing_page_v3'),
    ('pricing:enterprise',    'Enterprise pricing is custom; contact sales@acme.io for a quote.', 'pricing_page_v3'),
    ('policy:refund',         'Acme offers a 30-day money-back guarantee on all self-serve plans.', 'terms_v12'),
    ('support:hours',         'Support is staffed 24x5, Monday through Friday, in IST and PST timezones.', 'support_runbook'),
    ('product:integrations',  'Acme integrates natively with Slack, Jira, Okta, and Google Workspace.', 'docs_v9'),
]
for topic, fact, source in ACME_FACTS:
    semantic.add_fact(topic, fact, source)

print('=== SEMANTIC QUERIES ===')
for q in ['how long do I have to return something?', 'do you integrate with Okta?', 'how much is the Pro tier?']:
    print(f'\nQ: {q}')
    for hit in semantic.query(q, k=2):
        print(f'  -> [{hit["topic"]}] {hit["fact"]}  (src={hit["source"]})')

=== SEMANTIC QUERIES ===

Q: how long do I have to return something?
  -> [policy:refund] Acme offers a 30-day money-back guarantee on all self-serve plans.  (src=terms_v12)
  -> [support:hours] Support is staffed 24x5, Monday through Friday, in IST and PST timezones.  (src=support_runbook)

Q: do you integrate with Okta?
  -> [product:integrations] Acme integrates natively with Slack, Jira, Okta, and Google Workspace.  (src=docs_v9)
  -> [pricing:enterprise] Enterprise pricing is custom; contact sales@acme.io for a quote.  (src=pricing_page_v3)

Q: how much is the Pro tier?
  -> [pricing:pro] The Pro plan costs $49 per seat per month and includes SSO and priority email support.  (src=pricing_page_v3)
  -> [pricing:starter] The Starter plan costs $9 per month and includes 3 seats.  (src=pricing_page_v3)


## Procedural memory: learned playbooks indexed by task signature

Now let's build procedural memory. Procedural = **learned strategies and workflows** - the agent's HOW-TO library. Concretely we store named prompt templates and ordered steps indexed by a *task signature* (a short trigger string).

Now, we will
- define a `ProceduralMemory` class wrapping a dict of named playbooks plus a vector store of triggers,
- expose `register(name, template, trigger)` for write and `lookup(task_signature)` for read,
- register three playbooks: `password_reset`, `escalate_to_billing`, `summarise_ticket`,
- look up the right playbook given a fresh user message and confirm semantic match on the trigger surfaces it.

In [8]:
PASSWORD_RESET_PROMPT = """You are a support agent. Run the password reset playbook:
1. Verify identity via the email on file.
2. Send a magic-link reset valid for 15 minutes.
3. Confirm successful login after the reset.
User context: {context}"""

BILLING_ESCALATION_PROMPT = """You are a support agent. Run the billing escalation playbook:
1. Collect invoice number and disputed amount.
2. Check refund eligibility against the 30-day policy.
3. If eligible, issue refund; else, route to a human billing agent.
User context: {context}"""

SUMMARISE_TICKET_PROMPT = """You are a support agent. Summarise this ticket in two sentences
for an engineering handoff. Capture symptom, environment, and expected vs actual.
User context: {context}"""

Three UPPER_CASE prompt constants - one per playbook. Module-level constants are easy to find, easy to diff, and easy to swap into other notebooks. The `{context}` placeholder is what `.format(context=...)` will fill at call time.

In [9]:
class ProceduralMemory:
    """Dictionary of named playbooks indexed by task signature.

    Procedural memory is what turns a one-off agent into a reliable one - the agent does NOT
    re-derive 'what do I do when X' every turn; it consults a runbook. WHY index by trigger:
    new user messages rarely match playbook NAMES, but they often match trigger DESCRIPTIONS."""

    def __init__(self, embedder):
        self._book: Dict[str, dict] = {}
        self._store = InMemoryVectorStore(embedding=embedder)

    def register(self, name: str, template: str, trigger: str) -> None:
        """Add a playbook under `name` with a `trigger` description used for semantic lookup."""
        self._book[name] = {'name': name, 'template': template, 'trigger': trigger}
        self._store.add_documents([Document(
            page_content=trigger,
            metadata={'name': name},
        )])

    def lookup(self, task_signature: str) -> Optional[dict]:
        """Return the best-matching playbook for a task signature, or None if empty."""
        if not self._book:
            return None
        hits = self._store.similarity_search(task_signature, k=1)
        if not hits:
            return None
        return self._book.get(hits[0].metadata['name'])

`lookup` returns the full playbook dict (name, template, trigger). The caller does the `.format()` and `.invoke()` - procedural memory is a retrieval problem first; execution stays downstream so you can swap retrieval without rewriting agents.

In [10]:
procedural = ProceduralMemory(embed)

procedural.register(
    name='password_reset',
    template=PASSWORD_RESET_PROMPT,
    trigger='User cannot log in, forgot password, MFA failure, needs a reset link.',
)
procedural.register(
    name='escalate_to_billing',
    template=BILLING_ESCALATION_PROMPT,
    trigger='User disputes a charge, wants a refund, confused about invoice or proration.',
)
procedural.register(
    name='summarise_ticket',
    template=SUMMARISE_TICKET_PROMPT,
    trigger='Internal handoff: write a short engineering-ready summary of a customer issue.',
)

print('=== PROCEDURAL LOOKUP ===')
for sig in ['I forgot my password and cannot log in.',
            'Please refund my last invoice.',
            'Write a summary of this bug for the eng team.']:
    pb = procedural.lookup(sig)
    print(f'\nTask: {sig}')
    print(f'  -> playbook: {pb["name"]}')
    print(f'     trigger : {pb["trigger"]}')

=== PROCEDURAL LOOKUP ===

Task: I forgot my password and cannot log in.
  -> playbook: password_reset
     trigger : User cannot log in, forgot password, MFA failure, needs a reset link.

Task: Please refund my last invoice.
  -> playbook: escalate_to_billing
     trigger : User disputes a charge, wants a refund, confused about invoice or proration.

Task: Write a summary of this bug for the eng team.
  -> playbook: summarise_ticket
     trigger : Internal handoff: write a short engineering-ready summary of a customer issue.


## Side-by-side: same question, three memory backings

Now let's contrast the three subtypes head-to-head. We will ask one realistic agent question and see what each memory returns - and crucially, **why each maps to a different question shape**.

Now, we will
- pose a single user question that touches all three memory subtypes,
- recall from `episodic`, `query` from `semantic`, `lookup` from `procedural`,
- read the three outputs as three answers to three different questions: *what happened?* vs *what is true?* vs *what do I do?*

In [11]:
USER_QUESTION = 'I had a CSV export bug last week, and now I want a refund. How do you handle this?'

ep_hits   = episodic.recall(USER_QUESTION, k=2)
sem_hits  = semantic.query(USER_QUESTION, k=2)
proc_hit  = procedural.lookup(USER_QUESTION)

print(f'---- USER QUESTION ----')
print(USER_QUESTION)

print(f'\n---- EPISODIC (what happened to this user before?) ----')
for ep in ep_hits:
    print(f'  [{ep.run_id}] {ep.query} -> {ep.outcome}')

print(f'\n---- SEMANTIC (what does the agent know about the world?) ----')
for hit in sem_hits:
    print(f'  [{hit["topic"]}] {hit["fact"]}')

print(f'\n---- PROCEDURAL (what should the agent do?) ----')
print(f'  playbook: {proc_hit["name"]}')
print(f'  trigger : {proc_hit["trigger"]}')

---- USER QUESTION ----
I had a CSV export bug last week, and now I want a refund. How do you handle this?

---- EPISODIC (what happened to this user before?) ----
  [run-2] User reports CSV export produces empty file -> Hotfix shipped; user verified working
  [run-1] User cannot reset password on Safari -> User confirmed login after 2 minutes

---- SEMANTIC (what does the agent know about the world?) ----
  [policy:refund] Acme offers a 30-day money-back guarantee on all self-serve plans.
  [pricing:enterprise] Enterprise pricing is custom; contact sales@acme.io for a quote.

---- PROCEDURAL (what should the agent do?) ----
  playbook: escalate_to_billing
  trigger : User disputes a charge, wants a refund, confused about invoice or proration.


Three memory subtypes, three question shapes. Episodic answered *recall an event*; semantic answered *look up a fact*; procedural answered *apply a procedure*. A real agent always needs all three on a non-trivial turn - missing any one of them produces a recognisable failure mode (re-asking the user, hallucinating policy, or improvising a workflow).

## Unified retrieval: one helper, all three stores

Finally, let's wire the three subtypes into a single agent call. The pattern is straightforward: one helper pulls from all three stores and merges the results into a structured prompt; the LLM produces a grounded answer.

Now, we will
- define `recall_all(question)` that returns a dict with `episodic`, `semantic`, `procedural` blocks,
- define a `unified_answer_node(question)` that formats those blocks into a single prompt and calls the LLM,
- invoke it on the same `USER_QUESTION` from the previous section,
- print the merged context AND the LLM's grounded reply under `=== ===` banners.

In [12]:
UNIFIED_PROMPT = """You are a grounded support agent. Use the three memory blocks below to answer.

EPISODIC (this user's history):
{episodic}

SEMANTIC (durable facts about Acme):
{semantic}

PROCEDURAL (the playbook to follow):
{procedural}

Question: {question}
Answer in 3-5 sentences, citing facts from the blocks above.
"""

def recall_all(question: str) -> dict:
    """Pull from all three memory stores and return a structured dict.

    The three retrievals are independent. WHY: each memory subtype has its own access pattern,
    and merging only at the prompt-building step keeps the stores swappable in isolation."""
    return {
        'episodic':   episodic.recall(question, k=2),
        'semantic':   semantic.query(question, k=3),
        'procedural': procedural.lookup(question),
    }

def _format_episodic(eps: List[Episode]) -> str:
    return '\n'.join(f'- [{ep.run_id}] {ep.query} -> {ep.outcome}' for ep in eps) or '(no prior episodes)'

def _format_semantic(facts: List[dict]) -> str:
    return '\n'.join(f'- {f["fact"]}' for f in facts) or '(no facts)'

def _format_procedural(pb: Optional[dict]) -> str:
    if not pb:
        return '(no matching playbook)'
    return f'playbook={pb["name"]} | template:\n{pb["template"]}'

def unified_answer_node(question: str) -> dict:
    """Build the unified prompt from all three memories and invoke the LLM once."""
    bundle = recall_all(question)
    prompt = UNIFIED_PROMPT.format(
        episodic=_format_episodic(bundle['episodic']),
        semantic=_format_semantic(bundle['semantic']),
        procedural=_format_procedural(bundle['procedural']),
        question=question,
    )
    return {'prompt': prompt, 'answer': llm.invoke(prompt).content}

One node, three retrievals, one LLM call. The merging happens at prompt construction time, not at storage time - that is what keeps episodic / semantic / procedural independently swappable in production. Now let's see it run.

In [13]:
result = unified_answer_node(USER_QUESTION)

print('=== MERGED PROMPT ===')
print(result['prompt'])
print('\n=== UNIFIED ANSWER ===')
print(result['answer'])

=== MERGED PROMPT ===
You are a grounded support agent. Use the three memory blocks below to answer.

EPISODIC (this user's history):
- [run-2] User reports CSV export produces empty file -> Hotfix shipped; user verified working
- [run-1] User cannot reset password on Safari -> User confirmed login after 2 minutes

SEMANTIC (durable facts about Acme):
- Acme offers a 30-day money-back guarantee on all self-serve plans.
- Enterprise pricing is custom; contact sales@acme.io for a quote.
- Acme integrates natively with Slack, Jira, Okta, and Google Workspace.

PROCEDURAL (the playbook to follow):
playbook=escalate_to_billing | template:
You are a support agent. Run the billing escalation playbook:
1. Collect invoice number and disputed amount.
2. Check refund eligibility against the 30-day policy.
3. If eligible, issue refund; else, route to a human billing agent.
User context: {context}

Question: I had a CSV export bug last week, and now I want a refund. How do you handle this?
Answer i

### Choosing the right memory subtype
- Use **episodic** when the question is *"what happened in run #42?"* or *"what did this user ask before?"*. The schema is event-shaped (timestamp, query, action, outcome) and the store is append-only.
- Use **semantic** when the question is *"what is true about the domain?"*. The schema is fact-shaped (topic, fact, source) and the store is upsert-keyed by topic so policy changes are a one-line edit.
- Use **procedural** when the question is *"how do I do X?"*. The schema is playbook-shaped (name, template, trigger) and lookup happens against the trigger description, not the playbook name.
- A non-trivial agent turn almost always needs all three. The integrated `recall_all(...)` helper is the canonical pattern - one call, three independent stores, one merged prompt.

### Production pitfalls
- `InMemoryVectorStore` is dev-only. Swap to `Chroma`, `FAISS`, or `pgvector` for durable storage; the `add_documents` / `similarity_search` interface is identical, so your three memory classes do not change.
- Episodic memory grows without bound. Schedule a summarisation job that compresses the oldest episodes into shorter summaries (and re-embeds them) so retrieval quality does not decay over time.
- Semantic facts go stale. Add a `last_verified` timestamp and reject any fact older than your policy refresh window when building the prompt - hallucinated old policy is the most common semantic-memory failure.
- Procedural lookup is only as good as the trigger string. If two playbooks have overlapping triggers, retrieval becomes coin-flippy; rewrite triggers to use distinctive language (verbs and entity nouns) instead of generic descriptions.